In [1]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /home/takayuki/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
from nltk.corpus import wordnet as wn

# The word we want to find synsets for
search_word = 'plant'

# Get all synsets for the word
synsets = wn.synsets(search_word, pos=wn.NOUN) # We only want nouns

if not synsets:
    print(f"No noun synsets found for '{search_word}'.")
else:
    print(f"Found {len(synsets)} synsets for '{search_word}.n':")
    # Loop through them and print their details
    for ss in synsets:
        print(f"- Name: '{ss.name()}'")
        print(f"  Definition: {ss.definition()}")
        print("-" * 20)

Found 4 synsets for 'plant.n':
- Name: 'plant.n.01'
  Definition: buildings for carrying on industrial labor
--------------------
- Name: 'plant.n.02'
  Definition: (botany) a living organism lacking the power of locomotion
--------------------
- Name: 'plant.n.03'
  Definition: an actor situated in the audience whose acting is rehearsed but seems spontaneous to the audience
--------------------
- Name: 'plant.n.04'
  Definition: something planted secretly for discovery by another
--------------------


In [3]:
import networkx as nx
from nltk.corpus import wordnet as wn
from nltk.corpus.reader.wordnet import WordNetError
import pygraphviz as pgv
import os

# --- MODIFIED HELPER FUNCTION ---
def get_wordnet_graph(start_synset_name, depth_down=1, depth_up=100):
    """
    Creates a NetworkX graph from a starting WordNet synset.
    This version correctly traverses downwards to the specified depth.
    """
    G = nx.DiGraph()
    start_synset = wn.synset(start_synset_name)

    # --- FIX: Implement a proper Breadth-First Search for downward traversal ---
    queue_down = [(start_synset, 0)]
    visited_down = set()
    
    while queue_down:
        current_synset, current_depth = queue_down.pop(0)

        if current_synset in visited_down or current_depth >= depth_down:
            continue
        
        visited_down.add(current_synset)
        
        for hyponym in current_synset.hyponyms():
            G.add_edge(current_synset.name(), hyponym.name())
            queue_down.append((hyponym, current_depth + 1))
    # --- END FIX ---

    # Upward traversal (Hypernyms) - this logic was already correct
    queue_up = [(start_synset, 0)]
    visited_up = set()
    while queue_up:
        current_synset, current_depth = queue_up.pop(0)
        if current_synset in visited_up or current_depth >= depth_up: continue
        visited_up.add(current_synset)
        for hypernym in current_synset.hypernyms():
            G.add_edge(hypernym.name(), current_synset.name())
            queue_up.append((hypernym, current_depth + 1))
            
    return G

def visualize_wordnet_hierarchy(word_or_synset_name, output_filename=None, depth_down=1, verbose='no-errors'):
    """
    Finds a word/synset, prints definitions of its children, creates its 
    hierarchy graph, and saves it as an image.
    """
    start_synset = None
    word_for_filename = word_or_synset_name
    
    try:
        start_synset = wn.synset(word_or_synset_name)
        print(f"Input '{word_or_synset_name}' identified as a direct synset.")
        word_for_filename = start_synset.lemmas()[0].name()
    except (WordNetError, ValueError):
        # print(f"Input '{word_or_synset_name}' not a synset, treating as a word...")
        synsets = wn.synsets(word_or_synset_name, pos=wn.NOUN)
        if not synsets:
            if verbose != 'no-errors':
                print(f"Error: No noun synsets found for '{word_or_synset_name}'.")
            return False
        else:
            print(f"Found {len(synsets)} noun synsets for '{word_or_synset_name}':")
            
        start_synset = synsets[0]

    start_synset_name = start_synset.name()
    print(f"Using '{start_synset_name}'. Definition: {start_synset.definition()}")

    
    hyponyms = start_synset.hyponyms()
    if hyponyms:
        print("\n--- Definitions of Children (Hyponyms) ---")
        for hypo in hyponyms:
            print(f"- {hypo.name()}: {hypo.definition()}")
        print("-------------------------------------------\n")
    
    
    if output_filename is None:
        safe_word = word_for_filename.replace(' ', '_').split('.')[0]
        output_filename = f"{safe_word}_d{depth_down}_hierarchy.png"

    # The rest of the function remains the same...
    supported_formats = ['png', 'jpg', 'jpeg', 'svg', 'pdf', 'gif']
    file_ext = os.path.splitext(output_filename)[1][1:].lower()
    if file_ext not in supported_formats:
        output_filename = os.path.splitext(output_filename)[0] + '.png'

    wordnet_graph = get_wordnet_graph(start_synset_name, depth_down=depth_down)
    
    if not wordnet_graph or not wordnet_graph.nodes():
        if depth_down > 0 and not hyponyms:
             print(f"Note: '{start_synset_name}' is a leaf node and has no children to show.")
        else:
             print("Graph could not be generated.")
        return

    A = pgv.AGraph(directed=True, strict=True, rankdir='LR', name=word_for_filename)

    for u, v in wordnet_graph.edges():
        label_u = u.split('.')[0].replace('_', ' ')
        label_v = v.split('.')[0].replace('_', ' ')
        A.add_edge(u, v)
        A.get_node(u).attr['label'] = label_u
        A.get_node(v).attr['label'] = label_v
    
    A.node_attr['shape'] = 'oval'
    A.node_attr['style'] = 'filled'
    A.node_attr['fillcolor'] = 'skyblue'
    A.node_attr['fontname'] = 'Helvetica'
    A.edge_attr['color'] = 'gray'
    A.edge_attr['arrowsize'] = 0.5

    try:
        start_node = A.get_node(start_synset_name)
        start_node.attr['fillcolor'] = 'lightgreen'
        start_node.attr['penwidth'] = '2.0'
    except KeyError:
        pass

    print(f"Generating graph (depth={depth_down})... Saving to '{output_filename}'")
    try:
        A.layout(prog='dot')
        A.draw(output_filename)
        print("Done.\n\n")
    except Exception as e:
        print(f"Error during rendering: {e}")
    
    return True

In [4]:
visualize_wordnet_hierarchy("plant.n.02", depth_down=2)

Input 'plant.n.02' identified as a direct synset.
Using 'plant.n.02'. Definition: (botany) a living organism lacking the power of locomotion

--- Definitions of Children (Hyponyms) ---
- autophyte.n.01: plant capable of synthesizing its own food from simple organic substances
- apomict.n.01: a plant that reproduces or is reproduced by apomixis
- perennial.n.01: (botany) a plant lasting for three seasons or more
- air_plant.n.01: plant that derives moisture and nutrients from the air and rain; usually grows on another plant but not parasitic on it
- non-flowering_plant.n.01: a plant that does not bear flowers
- vascular_plant.n.01: green plant having a vascular system: ferns, gymnosperms, angiosperms
- endemic.n.02: a plant that is native to a certain limited area
- plantlet.n.01: a young plant or a small plant
- aquatic.n.01: a plant that lives in or on water
- annual.n.01: (botany) a plant that completes its entire life cycle within the space of a year
- myrmecophyte.n.01: plant that 

True

In [10]:
visualize_wordnet_hierarchy("aquatic.n.01", depth_down=2)
visualize_wordnet_hierarchy("hydrophyte.n.01", depth_down=1)

Input 'aquatic.n.01' identified as a direct synset.
Using 'aquatic.n.01'. Definition: a plant that lives in or on water
Generating graph (depth=2)... Saving to 'aquatic_d2_hierarchy.png'
Done.


Input 'hydrophyte.n.01' identified as a direct synset.
Using 'aquatic_plant.n.01'. Definition: a plant that grows partly or wholly in water whether rooted in the mud, as a lotus, or floating without anchorage, as the water hyacinth

--- Definitions of Children (Hyponyms) ---
- water_lily.n.01: an aquatic plant of the family Nymphaeaceae
- pondweed.n.01: any of several submerged or floating freshwater perennial aquatic weeds belonging to the family Potamogetonaceae
- american_frogbit.n.01: American plant with roundish heart-shaped or kidney-shaped leaves; usually rooted in muddy bottoms of ponds and ditches
- water_star_grass.n.01: grassy-leaved North American aquatic plant with yellow star-shaped blossoms
- naiad.n.01: submerged aquatic plant having narrow leaves and small flowers; of fresh or 

True

In [6]:
import os
DATA_DIR = "/home/takayuki/Desktop/summer2025/plants/data/AquaticPlantLabData/squared"
species_names = os.listdir(DATA_DIR)
common_names = [
    "Eurasian watermilfoil",
    "watershield",
    "white water crowfoot",
    "Illinois pondweed",
    "European frog bit",
    "yellow pond lily",
    "American white water lily",
    "Richardsons pondweed",
    "slender naiad",
    "Robbins pondweed",
    "American eelgrass",
    "curly leaf pondweed",
    "varied leaf pondweed",
    "Canadian waterweed",
    "fanwort",
    "northern watermilfoil",
    "long beaked pondweed",
    "coontail",
    "water stargrass",
    "starry stonewort",
    "broad leaf pondweed"
]
species_words = []

species_names.extend(common_names) 
for name in species_names:
    words = name.split(' ')
    species_words.extend(words)
species_words = list(set(species_words)) 


In [7]:
def find_synset(word_or_synset_name):
    """
    Helper function to find the most common noun synset for a word or use a direct synset name.
    Returns a Synset object or None if not found.
    """
    try:
        # First, try to treat the input as a specific synset name
        return wn.synset(word_or_synset_name)
    except (WordNetError, ValueError):
        # If that fails, treat it as a regular word to search for
        synsets = wn.synsets(word_or_synset_name, pos=wn.NOUN)
        if not synsets:
            return None
        return synsets[0] # Use the most common sense

def is_descendant_of(synset_to_check, parent_synset):
    """
    Checks if a synset is a descendant of a given parent synset.

    Args:
        synset_to_check (Synset): The potential descendant synset.
        parent_synset (Synset): The potential ancestor synset.

    Returns:
        bool: True if synset_to_check is a descendant of parent_synset, False otherwise.
    """
    if not synset_to_check or not parent_synset:
        return False
        
    all_paths = synset_to_check.hypernym_paths()
    
    for path in all_paths:
        if parent_synset in path:
            return True
            
    return False

plant_hits = []
plants_count = 0
all_count = 0
for word in species_words:
    synset = find_synset(word)
    if synset:
        all_count += 1
        if is_descendant_of(synset, wn.synset('plant.n.02')):
            plant_hits.append(word)
            plants_count += 1
print(f"Total words matched: {all_count}")
print(f"Found {len(plant_hits)} words that are descendants of 'plant.n.02':")

for hit in plant_hits:
    print(f"- {hit}")


Total words matched: 38
Found 7 words that are descendants of 'plant.n.02':
- waterweed
- naiad
- crowfoot
- eelgrass
- lily
- fanwort
- pondweed


In [8]:
print("Visualizing them")
visualize_wordnet_hierarchy(hit, depth_down=2, verbose='errors')


Visualizing them
Found 2 noun synsets for 'pondweed':
Using 'pondweed.n.01'. Definition: any of several submerged or floating freshwater perennial aquatic weeds belonging to the family Potamogetonaceae

--- Definitions of Children (Hyponyms) ---
- frog's_lettuce.n.01: very similar to Potamogeton; of western Africa, Asia, and Europe
- curled_leaf_pondweed.n.01: European herb naturalized in the eastern United States and California
- horned_pondweed.n.01: found in still or slow-moving fresh or brackish water; useful to oxygenate cool water ponds and aquaria
- variously-leaved_pondweed.n.01: of Europe (except the Mediterranean area) and the northern United States
- loddon_pondweed.n.01: pondweed with floating leaves; of northern United States and Europe
-------------------------------------------

Generating graph (depth=2)... Saving to 'pondweed_d2_hierarchy.png'
Done.




True